# MTH407 Term Project — 2D Target Localization & Tracking

Çoklu sensörler ile zaman tabanlı hedef lokalizasyonu ve takibi.

- **Measurement model:** Time of Arrival (TOA), $z_i^k = t_k + \|p^k - s_i\| / c + n_i^k$, $n \sim \mathcal{N}(0, \sigma_t^2)$.
- **Constants:** $c = 3\times10^8$ m/s, $\sigma_t = 10^{-8}$ s ⇒ effective range noise $\sigma_r = c\sigma_t = 3$ m.
- **Target trajectory:** zig-zag with 4 sharp heading changes inside a 10 km × 10 km arena, target speed 100 m/s, sample period $T = 0.5$ s, duration 60 s.
- **Estimators:** linearized least-squares (LSE) for state initialization; CV-EKF (constant-velocity Extended Kalman Filter) for sequential tracking.
- **Geometries compared:** $N \in \{2, 3, 4\}$, two geometries each (favourable / degraded), six cells total.

All figures and tables produced here also land in `output/figures/` and `output/results/`.

## 0. Setup

In [ ]:
%matplotlib inline
import sys, pathlib
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src import C_LIGHT, SIGMA_T, SIGMA_R, SAMPLE_PERIOD, ARENA_SIZE, TARGET_SPEED
from src.trajectory import make_zigzag
from src.sensors import GEOMETRIES, GEOMETRY_LABELS, get_geometry
from src.measurements import simulate_toa, to_range
from src.lse import lse_position, lse_two_sensor, lse_track, initialize_state
from src.ekf import run_ekf, cv_F, cv_Q
from src.metrics import position_error

print(f'c = {C_LIGHT:.0e} m/s   sigma_t = {SIGMA_T:.0e} s   sigma_r = {SIGMA_R:.1f} m')
print(f'arena = {ARENA_SIZE/1000:.0f} km   target speed = {TARGET_SPEED:.0f} m/s   T = {SAMPLE_PERIOD:.2f} s')

## 1. Ground-truth trajectory

Five constant-velocity legs joined by four sharp heading changes. Position and velocity at each sample step are stacked in a $(K, 4)$ array as $[p_x, p_y, v_x, v_y]$.

In [ ]:
states_truth, times = make_zigzag()
K = len(times)
print(f'K = {K} steps,   t[0] = {times[0]:.1f}s,   t[-1] = {times[-1]:.1f}s')
print(f'speed (constant) = {np.linalg.norm(states_truth[0,2:]):.1f} m/s')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(states_truth[:, 0], states_truth[:, 1], 'k-', lw=2)
ax.plot(*states_truth[0, :2], 'ko', ms=8, label='start')
ax.plot(*states_truth[-1, :2], 'ks', ms=8, label='end')
for ts in [24, 48, 72, 96]:
    ax.plot(*states_truth[ts, :2], 'rx', ms=10)
ax.set_xlim(0, ARENA_SIZE); ax.set_ylim(0, ARENA_SIZE)
ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.set_title('Ground-truth zig-zag (turns marked with x)')
ax.legend(); plt.show()

## 2. Sensor geometries

Six configurations span $N \in \{2, 3, 4\}$ with one favourable and one degraded layout each.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, name in zip(axes.flatten(), list(GEOMETRIES)):
    sensors = get_geometry(name)
    ax.plot(states_truth[:, 0], states_truth[:, 1], 'k-', lw=1.2, alpha=0.6)
    ax.scatter(sensors[:, 0], sensors[:, 1], marker='^', s=100, c='red', edgecolors='black', zorder=5)
    ax.set_xlim(-500, ARENA_SIZE+500); ax.set_ylim(-500, ARENA_SIZE+500)
    ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.set_title(GEOMETRY_LABELS[name], fontsize=10)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
fig.tight_layout(); plt.show()

## 3. TOA measurement model

For each sample time $t_k$ and sensor $i$ we generate a noisy time-of-arrival, then convert to an equivalent range with $r_i^k = c (z_i^k - t_k)$. The range residuals follow $\mathcal{N}(0, \sigma_r^2)$ with $\sigma_r = c\sigma_t = 3$ m.

In [ ]:
sensors = get_geometry('N4_square')
rng = np.random.default_rng(42)
z, emit = simulate_toa(states_truth, sensors, rng=rng)
r = to_range(z, emit)
true_dists = np.linalg.norm(states_truth[:, None, :2] - sensors[None, :, :], axis=2)
residuals = r - true_dists
print(f'Noise std (empirical): {residuals.std():.3f} m   (expected {SIGMA_R:.3f} m)')
print(f'Noise mean (empirical): {residuals.mean():.4f} m')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(residuals.flatten(), bins=40, density=True, alpha=0.7)
x_grid = np.linspace(-12, 12, 200)
ax.plot(x_grid, np.exp(-x_grid**2/(2*SIGMA_R**2))/(SIGMA_R*np.sqrt(2*np.pi)), 'r-', lw=1.5, label='N(0, σ_r²)')
ax.set_xlabel('range residual [m]'); ax.set_ylabel('density'); ax.legend(); ax.grid(alpha=0.3); plt.show()

## 4. LSE initialization

For $N \geq 3$ we iterate Gauss-Newton on the non-linear range model. For $N = 2$ we solve the two-circle intersection in closed form and disambiguate with a coarse prior. The initial state vector $[p_x, p_y, v_x, v_y]^\top$ uses the first two time steps; velocity comes from finite difference.

In [ ]:
PRIOR_P = np.array([3000., 4000.])
for name in GEOMETRIES:
    sensors = get_geometry(name)
    rng = np.random.default_rng(7)
    z, emit = simulate_toa(states_truth, sensors, rng=rng)
    r = to_range(z, emit)
    x0, P0 = initialize_state(r[:2], sensors, T=SAMPLE_PERIOD, prior_p=PRIOR_P)
    err = np.linalg.norm(x0[:2] - states_truth[0, :2])
    print(f'{GEOMETRY_LABELS[name]:<32s} pos err {err:7.2f} m   v_init=({x0[2]:+6.2f},{x0[3]:+6.2f}) m/s')

## 5. EKF baseline (N = 4 square)

Process model: 2D constant velocity with discrete white-acceleration $Q(\sigma_a)$.
Tuned $\sigma_a = 10$ m/s² balances tracking responsiveness with steady-state smoothness given the sharp zig-zag turns.

In [ ]:
from experiments.exp_baseline import run_baseline
res = run_baseline()
err_ekf = position_error(res['states_ekf'], res['truth'])
err_lse = np.linalg.norm(res['lse_positions'] - res['truth'][:, :2], axis=1)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios':[2.4, 1]})
ax = axes[0]
ax.plot(res['truth'][:, 0], res['truth'][:, 1], 'k-', lw=2.0, label='truth')
ax.plot(res['lse_positions'][:, 0], res['lse_positions'][:, 1], 'C3.', ms=4, alpha=0.6, label='LSE per-step')
ax.plot(res['states_ekf'][:, 0], res['states_ekf'][:, 1], 'C0-', lw=1.5, label='EKF')
ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_title(f'Trajectory zoom — {GEOMETRY_LABELS[res["geometry"]]}'); ax.legend()
ax = axes[1]
ax.plot(res['times'], err_lse, 'C3-', lw=1.2, label='LSE')
ax.plot(res['times'], err_ekf, 'C0-', lw=1.2, label='EKF')
for ts in [12, 24, 36, 48]:
    ax.axvline(ts, color='gray', ls=':', alpha=0.6)
ax.set_xlabel('time [s]'); ax.set_ylabel('position error [m]'); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

rmse_ekf = np.sqrt(np.mean(err_ekf[5:]**2))
rmse_lse = np.sqrt(np.mean(err_lse[5:]**2))
print(f'Overall: LSE RMSE = {rmse_lse:.2f} m,   EKF RMSE = {rmse_ekf:.2f} m')

**Observation.** Between turns the EKF tightens below the per-step LSE (steady-state ≈ $\sigma_r/\sqrt{N}$). At each of the four turn instants the CV model is briefly violated, producing a transient spike up to ~17 m. With smoother (coordinated) turns, or a higher-order motion model (CTR, IMM), these spikes would vanish.

## 6. Sensor-geometry sweep

Same trajectory, same noise seed, six geometry cells.

In [ ]:
from experiments.exp_geometry import CELL_ORDER, run_one
results = {n: run_one(n) for n in CELL_ORDER}
rows = []
for n in CELL_ORDER:
    r = results[n]
    rows.append((GEOMETRY_LABELS[n], r['rmse_lse'], r['rmse_ekf']))
print(f'{"cell":<32s}{"LSE RMSE":>12s}{"EKF RMSE":>12s}')
print('-'*56)
for label, lse, ekf in rows:
    print(f'{label:<32s}{lse:12.2f}{ekf:12.2f}')

**Read-off.**
- The two best-conditioned cells (`N=4 square`, `N=3 triangle`) sit within a few metres of the theoretical bound $\sigma_r/\sqrt{N}$.
- `N=3 collinear` is competitive only because the target stays well off the sensor line; if it crossed the line, the rank deficiency would dominate.
- `N=2 widespread` has bimodal LSE (the two circle intersections), but the EKF disambiguates over time and recovers a 25–30 m track.
- `N=2 close pair` is fundamentally degenerate: the perpendicular-to-baseline direction is poorly observable, and the EKF cannot recover.
- `N=4 cluster` packs four sensors into a 500 m corner cell, so range gradients are nearly co-linear → high GDOP (~30–40 m RMSE).

## 7. Monte Carlo (100 trials per cell)

In [ ]:
from experiments.exp_monte_carlo import run_mc_for_cell, CELL_ORDER as MC_ORDER
mc = {n: run_mc_for_cell(n) for n in MC_ORDER}
print(f'{"cell":<32s}{"LSE mean ± std":>22s}{"EKF mean ± std":>22s}')
print('-'*78)
for n in MC_ORDER:
    d = mc[n]
    print(f'{GEOMETRY_LABELS[n]:<32s}{d["rmse_lse_mean"]:10.2f} ± {d["rmse_lse_std"]:5.2f}    {d["rmse_ekf_mean"]:10.2f} ± {d["rmse_ekf_std"]:5.2f}')

In [ ]:
labels = [GEOMETRY_LABELS[n] for n in MC_ORDER]
rmse_lse = [mc[n]['rmse_lse_mean'] for n in MC_ORDER]
rmse_ekf = [mc[n]['rmse_ekf_mean'] for n in MC_ORDER]
rmse_lse_std = [mc[n]['rmse_lse_std'] for n in MC_ORDER]
rmse_ekf_std = [mc[n]['rmse_ekf_std'] for n in MC_ORDER]
x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x-w/2, rmse_lse, w, yerr=rmse_lse_std, capsize=4, color='C3', alpha=0.85, label='LSE per-step')
ax.bar(x+w/2, rmse_ekf, w, yerr=rmse_ekf_std, capsize=4, color='C0', alpha=0.85, label='EKF')
bound = SIGMA_R / np.sqrt(np.array([2,2,3,3,4,4]))
ax.plot(x, bound, 'k--', lw=1.0, alpha=0.7, label='σ_r/√N')
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('position RMSE [m] (log)'); ax.set_title('100-trial MC summary'); ax.grid(axis='y', which='both', alpha=0.3)
ax.legend(loc='upper left'); fig.tight_layout(); plt.show()

## 8. Conclusions

1. **Sensor count alone is not enough.** A 4-sensor cluster (N=4 cluster, ~40 m RMSE) is decisively beaten by a 3-sensor equilateral triangle (~3.5 m). Geometry beats raw count.
2. **The EKF is not a substitute for adequate geometry.** For N=2 close-pair the perpendicular-to-baseline direction is essentially unobservable; sequential filtering does not unlock it.
3. **Per-step LSE vs EKF.** With sharp instantaneous heading changes, a CV-EKF pays a turn-induced spike penalty (~10–17 m here). Within a smooth leg the EKF averages measurements over time and beats per-step LSE by ~15–25%. Over the full 60 s run the two approaches end up within a small factor of each other for well-conditioned geometries.
4. **The N=2 widespread cell is the most interesting EKF win:** per-step LSE is bimodal and gives ~570 m RMSE, but the EKF resolves the ambiguity over time and tracks at ~28 m.
5. **Recommendation for the spec's question:** the canonical N=4 square layout is best. Equilateral N=3 is the cheapest acceptable alternative. Avoid clustered or collinear deployments unless geometry-specific priors are available.